# 2025年8月韩国海关前20日原稿：单次TLS传输修复

只重试固定官方列表URL一次；其余44个留出月份原样保留，不读取SOXX结果。


In [ ]:
import hashlib,json,os,pathlib,tarfile,tempfile,subprocess,shlex,urllib.request,urllib.parse,urllib.error,datetime,re
import pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.workflow import R
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2025_aug_transport_repair_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
old=root/src['held_capture_card'];artifact=old/'qlib_recorder'/src['held_capture_experiment_id']/src['held_capture_recorder_id']/'artifacts/analysis/monthly_capture';year_manifest=root/'data/snapshots/kcs_early20_2023_2026_held_original_capture_20261023/2025/year_manifest.json'
for p,h in [(old/'summary.json',src['held_capture_summary_sha256']),(artifact,src['held_capture_monthly_artifact_sha256']),(year_manifest,src['prior_2025_year_manifest_sha256'])]:assert sha(p)==h
old_aug=next(x for x in json.loads(year_manifest.read_text())['months'] if x['month']=='2025-08');assert old_aug['list']['body_bytes']==0 and old_aug['list']['http_status'] is None and 'handshake operation timed out' in old_aug['list']['error']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(old/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['held_capture_experiment_id'],recorder_id=src['held_capture_recorder_id']);assert rec.status=='FINISHED';table=rec.load_object('analysis/monthly_capture').copy()
assert len(table)==45 and table.loc[table.month.eq('2025-08'),'list_status'].iloc[0]=='transport_unknown' and table.loc[table.month.ne('2025-08'),'all_linked_signatures_qualified'].all()
q=plan['transport'];snap=root/'data/snapshots/kcs_early20_2025_aug_transport_repair_20261023';snap.mkdir(parents=True,exist_ok=True);state={'status':'one_list_GET_started','plan_sha256':sha(card/'plan.json'),'list_attempts':0,'page_attempts':0,'attachment_attempts':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def get(url,maxbytes):
 start=datetime.datetime.now(datetime.timezone.utc).isoformat();headers={};body=b'';status=None;error=None;final=url
 try:
  r=urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':q['user_agent']}),timeout=q['timeout_seconds_each']);status=r.status;final=r.url;headers={k:v for k,v in r.headers.items() if k.lower() not in ('set-cookie','set-cookie2','authorization','proxy-authenticate')};body=r.read(maxbytes+1);r.close()
 except urllib.error.HTTPError as exc:status=exc.code;final=exc.url;headers={k:v for k,v in exc.headers.items() if k.lower() not in ('set-cookie','set-cookie2','authorization','proxy-authenticate')};body=exc.read(maxbytes+1);error='HTTPError'
 except Exception as exc:error=type(exc).__name__+': '+str(exc)[:180]
 return {'url':url,'final_url':final,'started_utc':start,'ended_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'http_status':status,'content_type':headers.get('Content-Type',headers.get('content-type','')),'body_sha256':hashlib.sha256(body).hexdigest(),'body_bytes':len(body),'error':error,'headers':headers},body
def save(key,kind,r,body):
 p=snap/(key+'.'+kind+'.response');p.write_bytes(body);h=snap/(key+'.'+kind+'.headers.json');h.write_text(json.dumps(r.pop('headers'),ensure_ascii=False,indent=2)+'\n');r['local_path']=str(p.relative_to(root));r['headers_path']=str(h.relative_to(root));r['headers_sha256']=sha(h);return r
state['list_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');listing,body=get(plan['exact_list_url'],q['max_list_bytes']);listing=save('2025-08','list',listing,body);accepted=[];candidates=[];list_status='transport_unknown'
if listing['http_status']==200 and listing['body_bytes']<=q['max_list_bytes'] and 'html' in listing['content_type'].lower():
 soup=BeautifulSoup(body,'html.parser')
 for a in soup.find_all('a'):
  href=a.get('href') or ''
  if 'pressReleaseView.do' not in href:continue
  name=a.get_text(' ',strip=True);compact=re.sub(r'\s+','',name);id=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('newsId') or [None])[0]
  if not id:continue
  item={'newsId':id,'title_excerpt':name[:240],'url':urllib.parse.urljoin(plan['exact_list_url'],href)};candidates.append(item)
  if ((('8월1' in compact and '20일' in compact) or re.search(r'8\.1[-~∼〜]?20',compact)) and '수출입현황' in compact and '관세청' in compact):accepted.append(item)
 list_status='unique' if len(accepted)==1 else ('ambiguous' if len(accepted)>1 else 'missing')
listing['status']=list_status;listing['candidates']=candidates;listing['accepted']=accepted;page=None;page_status='not_attempted';links=[];attachments=[]
if list_status=='unique':
 identity=accepted[0]['newsId'];url='https://www.korea.kr/briefing/pressReleaseView.do?newsId='+identity;state['page_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');page,body=get(url,q['max_page_bytes']);page=save('2025-08','page',page,body);page_status='transport_unknown';page['newsId']=identity
 if page['http_status']==200 and page['body_bytes']<=q['max_page_bytes'] and 'html' in page['content_type'].lower():
  soup=BeautifulSoup(body,'html.parser');titles=[h.get_text(' ',strip=True) for h in soup.find_all('h1')];page['headings']=titles[:5];page_status='identified' if any('수출입' in h and ('20일' in h or re.search(r'8\.1[-~∼〜]?20',h)) for h in titles) else 'title_unknown'
  if page_status=='identified':
   seen=set()
   for a in soup.find_all('a'):
    href=a.get('href') or '';name=a.get_text(' ',strip=True);ext=re.search(r'\.(hwp|hwpx|pdf)\b',name,re.I);fileid=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('fileId') or [None])[0]
    if 'download.do' not in href or ext is None or fileid is None or fileid in seen:continue
    seen.add(fileid);links.append({'fileId':fileid,'filename':name,'ext':ext.group(1).lower(),'url':urllib.parse.urljoin(url,href)})
  if page_status=='identified' and len(links)<=4:
   for link in links:
    state['attachment_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');r,blob=get(link['url'],q['max_attachment_bytes']);r=save('2025-08_'+link['fileId'],'attachment',r,blob);r.update(link);magic={'hwp':bytes.fromhex('d0cf11e0a1b11ae1'),'hwpx':b'PK\x03\x04','pdf':b'%PDF'}[link['ext']];r['signature_ok']=bool(r['http_status']==200 and r['body_bytes']<=q['max_attachment_bytes'] and blob.startswith(magic));attachments.append(r)
  elif len(links)>4:page_status='attachment_overflow_unknown'
 page['status']=page_status;page['attachment_links']=links;page['attachments']=attachments
qualified=bool(links and len(links)==len(attachments) and all(a['signature_ok'] for a in attachments));new={'month':'2025-08','newsId':accepted[0]['newsId'] if len(accepted)==1 else None,'list_status':list_status,'page_status':page_status,'attachment_links':len(links),'attachment_attempts':len(attachments),'signature_qualified_attachments':sum(x['signature_ok'] for x in attachments),'all_linked_signatures_qualified':qualified};loc=table.month.eq('2025-08');assert loc.sum()==1
if qualified:
 for k,v in new.items():table.loc[loc,k]=v
assert int(table.loc[table.month.ne('2025-08'),'all_linked_signatures_qualified'].sum())==44 and len(table)==45
manifest={'plan_sha256':sha(card/'plan.json'),'prior_zero_body_TLS_error':{k:old_aug['list'][k] for k in ('started_utc','ended_utc','error','body_sha256','body_bytes')},'new_list':listing,'new_page':page,'new_month_result':new};(snap/'repair_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n');files=sorted(p for p in snap.iterdir() if p.is_file());settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-aug-repair-') as temp:
 archive=pathlib.Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size;remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest;source_archive={**receipt,'nas_path':remote,'files':len(files),'repair_manifest_sha256':sha(snap/'repair_manifest.json')}
 with tempfile.TemporaryDirectory(prefix='kcs-aug-cold-') as tmp:
  copy=pathlib.Path(tmp)/'source.tar.gz'
  with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True)
  assert copy.stat().st_size==receipt['size_bytes'] and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as t:
   names={m.name for m in t if m.isfile()};assert names=={str(p.relative_to(root)) for p in files}
   for p in files:assert t.extractfile(str(p.relative_to(root))).read()==p.read_bytes()
(card/'source_restore.json').write_text(json.dumps({'status':'NAS_2025_Aug_repair_all_members_cold_restored_exact','archive_sha256':digest,'files':len(files)},indent=2)+'\n');result={'status':'KCS_2025_Aug_one_exact_TLS_transport_repair_source_only','plan_sha256':sha(card/'plan.json'),'prior_unique_months':44,'new_month':new,'final_unique_months':int(table.all_linked_signatures_qualified.sum()),'unchanged_other_months':44,'list_GET_attempts':state['list_attempts'],'page_GET_attempts':state['page_attempts'],'attachment_GET_attempts':state['attachment_attempts'],'source_archive':source_archive,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,prior_monthly_sha256=src['held_capture_monthly_artifact_sha256'],source_archive=source_archive);rec.save_objects(artifact_path='analysis',repaired_held_index=table,August_receipts=manifest,result=result);rec.log_metrics(months=45,qualified_months=result['final_unique_months'],external_GETs=state['list_attempts']+state['page_attempts']+state['attachment_attempts'],model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'KCS_2025_Aug_held_source_transport_repaired_original_numeric_PIT_pending' if qualified else 'KCS_2025_Aug_held_source_transport_still_unknown','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':source_archive,'source_restore':'source_restore.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'verified':True},'final_unique_months':result['final_unique_months'],'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/repaired_held_index','analysis/August_receipts','analysis/result']},'limitations':['One repaired current download does not prove original 2025 first-public bytes.','Numeric HWPX/PDF value, working-day and decision-time qualifications are still pending.','No Stage1 factor, SOXX performance or funded account.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'month':new,'prior':44,'final':result['final_unique_months'],'source_sha256':digest,'recorder_id':rid,'NAS_sha256':evidence['sha256']},ensure_ascii=False,indent=2))


## 中文结论

前一固定留出来源卡的该月官方列表请求在TLS握手超时后结束，HTTP状态未知、正文 **0字节**；它不是“该月未发布”的证据。本卡按冻结的**同一目录URL仅重试一次**，得到唯一官方海关原稿 `newsId=156712088`，页面标题身份合格，所列 **2份 HWPX/PDF** 均为HTTP200且文件签名合格。其余44个月的已保存 Qlib索引逐行不变；45个月现均有官方页面及附件来源。

本次列表、页面、两份附件、响应头、UTC收取时钟及SHA原字节已归档Yanxuan NAS，源SHA-256 `6f088eb1d4dc8fd4a1affe91bb52a21bc1b936fe0189d3fc7de6a37fcfa009ea`，所有成员冷恢复一致；新原生Qlib Recorder `4a5ddf3df4d64e90b5eb6a8837e858b7` 的45行索引和失败/修复收据在NAS SHA-256 `cafded318019a5224fe20758c6f3a9eaeb8c79261f4cdbf44631b76f0a89692a` 下定向冷读一致。外部GET实际为列表1、页面1、附件2；**0 SOXX收益读取、0拟合、0资金账户**。

决定 `KCS_2025_Aug_held_source_transport_repaired_original_numeric_PIT_pending`。一次今天的成功续取不认证2025年8月首次公开的同一字节。45/45是留出月份原件传输覆盖；仍需将HWP、HWPX和PDF的半导体同比、工作日差异及历史首次可用时点分别核实，之后才可进入Stage1信息与完整0bp Tiered账户。稳定超过SOXX尚未得到证明。
